In [1]:
import numpy as np
import pandas as pd
import json
import os
import math
from scipy.stats import median_abs_deviation, t as t_dist
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append('.')
from mcv1_forecast.core import (train_model, recursive_forecast, 
                                engineer_features, COUNTRIES, TARGET, mape)
from sklearn.metrics import mean_absolute_error

In [2]:
print("Loading data and training baseline model...")
backend_dir = os.getcwd()
df_raw = pd.read_csv("vaccine_data.csv")
future_demo_df = pd.read_csv("future_demographics.csv")

model, df_engineered, feature_cols, dummy_cols = train_model(df_raw)

Loading data and training baseline model...


TimeSeries CV MAE Scores: [23.14338368  8.49470305 40.77694245]
Average CV MAE: 24.138


In [3]:
print("Running backtest (2020-2024)...")
backtest_raw = recursive_forecast(df_engineered, model, feature_cols, dummy_cols, 2020, TARGET)
backtest_data = {}
for c in COUNTRIES:
    c_back = backtest_raw[backtest_raw['Country'] == c].dropna(subset=['Actual', 'Predicted'])
    backtest_data[c] = c_back[['Year', 'Actual', 'Predicted']].rename(columns={
        'Year': 'year', 'Actual': 'actual', 'Predicted': 'predicted'
    }).to_dict('records')

Running backtest (2020-2024)...

--- Walk-forward retraining for year 2020 ---


TimeSeries CV MAE Scores: [11.75333295 25.1193557   6.75771992]
Average CV MAE: 14.543



--- Walk-forward retraining for year 2021 ---


TimeSeries CV MAE Scores: [12.14957139 21.36770506  6.5441483 ]
Average CV MAE: 13.354



--- Walk-forward retraining for year 2022 ---


TimeSeries CV MAE Scores: [12.49991606 24.56606677  9.39833966]
Average CV MAE: 15.488



--- Walk-forward retraining for year 2023 ---


TimeSeries CV MAE Scores: [13.01683638 13.77429642  5.29116162]
Average CV MAE: 10.694



--- Walk-forward retraining for year 2024 ---


TimeSeries CV MAE Scores: [22.93377554 11.36124033 15.76928509]
Average CV MAE: 16.688


In [4]:
print("Running forecast baseline (2025-2030)...")
forecast_raw = recursive_forecast(df_engineered, model, feature_cols, dummy_cols, 2025, TARGET, future_demo_df)
forecast_baseline = {}
for c in COUNTRIES:
    f_df = forecast_raw[forecast_raw['Country'] == c].sort_values('Year')
    forecast_baseline[c] = f_df['Predicted'].round(3).tolist()

Running forecast baseline (2025-2030)...


In [5]:
print("Running Monte Carlo simulations...")
class MonteCarloEngine:
    def __init__(self, model, df_engineered, feature_cols, dummy_cols,
                 n_simulations=500, random_seed=42):
        self.model = model
        self.df_engineered = df_engineered
        self.feature_cols = feature_cols
        self.dummy_cols = dummy_cols
        self.n_simulations = n_simulations
        self.rng = np.random.RandomState(random_seed)
        self.country_scale = {}
        self.pooled_residuals = np.array([])
        self._prepare_residual_pool()
        
    def _prepare_residual_pool(self):
        backtest_results = recursive_forecast(
            self.df_engineered, self.model, self.feature_cols,
            self.dummy_cols, 2010, TARGET
        )
        backtest_results = backtest_results.dropna(subset=['Actual', 'Predicted'])
        backtest_results['Residual'] = backtest_results['Actual'] - backtest_results['Predicted']
        
        all_standardized = []
        for country in COUNTRIES:
            res = backtest_results[backtest_results['Country'] == country]['Residual'].values
            if len(res) > 1:
                mad = median_abs_deviation(res, scale='normal')
                if mad == 0:
                    mad = max(np.max(np.abs(res)), 1e-6)
                med = np.median(res)
                self.country_scale[country] = mad
                z = (res - med) / mad
                all_standardized.append(z)
            else:
                self.country_scale[country] = 5.0
        
        if len(all_standardized) > 0:
            self.pooled_residuals = np.concatenate(all_standardized)
        else:
            self.pooled_residuals = np.array([0.0])

    def _generate_noise(self, n, country):
        from scipy.stats import t as t_dist
        pool = self.pooled_residuals
        if len(pool) < 2:
            return self.rng.normal(0, 1, size=n) * self.country_scale[country]
        df_t, loc_t, scale_t = t_dist.fit(pool)
        samples = t_dist.rvs(df_t, loc=loc_t, scale=scale_t, size=n, random_state=self.rng)
        return samples * self.country_scale[country]

    def perturb_demographics(self, demo_df):
        perturbed_df = demo_df.copy()
        years = perturbed_df['Year'].unique()
        base_year = min(years) if len(years) > 0 else 2025
        for idx, row in perturbed_df.iterrows():
            year_val = row['Year']
            year_idx = year_val - base_year
            horizon_scale = 1.0 + 0.15 * year_idx
            births_mult = self.rng.normal(1.0, 0.05 * horizon_scale)
            imr_mult = self.rng.normal(1.0, 0.10 * horizon_scale)
            
            perturbed_df.at[idx, 'Births (thousands)'] *= births_mult
            perturbed_df.at[idx, 'Pop_Age_0(In Thousands)'] *= births_mult
            perturbed_df.at[idx, 'Infant Mortality Rate (infant deaths per 1,000 live births)'] *= imr_mult
            perturbed_df.at[idx, 'Under-Five Mortality (deaths under age 5 per 1,000 live births)'] *= imr_mult
        return perturbed_df

    def run_simulations(self, future_demo_df):
        all_results = []
        for sim_id in range(self.n_simulations):
            sim_demo_df = self.perturb_demographics(future_demo_df)
            sim_pred = recursive_forecast(
                self.df_engineered, self.model, self.feature_cols,
                self.dummy_cols, 2025, TARGET, sim_demo_df
            )
            for country in COUNTRIES:
                country_mask = sim_pred['Country'] == country
                n_years = country_mask.sum()
                if n_years > 0:
                    noise = self._generate_noise(n_years, country)
                    last_actual = df_raw[df_raw['Country'] == country][TARGET].iloc[-1]
                    upper_bound = last_actual * 2.0
                    sim_pred.loc[country_mask, 'Predicted'] += noise
                    sim_pred.loc[country_mask, 'Predicted'] = sim_pred.loc[
                        country_mask, 'Predicted'
                    ].clip(lower=0, upper=upper_bound)
            sim_pred['Simulation_ID'] = sim_id
            all_results.append(sim_pred)
        return pd.concat(all_results, ignore_index=True)

    def compute_percentiles(self, sim_results):
        percentiles = [5, 10, 25, 50, 75, 90, 95]
        def pct_func(p): return lambda x: np.percentile(x, p)
        agg_funcs = {f'P{p}': pct_func(p) for p in percentiles}
        agg_funcs['Mean'] = 'mean'
        return sim_results.groupby(['Country', 'Year'])['Predicted'].agg(**agg_funcs).reset_index()

engine = MonteCarloEngine(model, df_engineered, feature_cols, dummy_cols, n_simulations=500)
sim_results = engine.run_simulations(future_demo_df)
percentiles_df = engine.compute_percentiles(sim_results)

mc_data = {}
for c in COUNTRIES:
    c_data = percentiles_df[percentiles_df['Country'] == c].sort_values('Year')
    mc_data[c] = [
        {
            "year": int(row['Year']),
            "p5": round(row['P5'], 1),
            "p25": round(row['P25'], 1),
            "p50": round(row['P50'], 1),
            "p75": round(row['P75'], 1),
            "p95": round(row['P95'], 1),
        }
        for _, row in c_data.iterrows()
    ]

Running Monte Carlo simulations...

--- Walk-forward retraining for year 2010 ---


TimeSeries CV MAE Scores: [14.35312367 14.83917534  9.4876618 ]
Average CV MAE: 12.893



--- Walk-forward retraining for year 2011 ---


TimeSeries CV MAE Scores: [17.42341432 10.66163086 24.69871182]
Average CV MAE: 17.595



--- Walk-forward retraining for year 2012 ---


...

TimeSeries CV MAE Scores: [12.49991606 24.56606677  9.39833966]
Average CV MAE: 15.488



--- Walk-forward retraining for year 2023 ---


TimeSeries CV MAE Scores: [13.01683638 13.77429642  5.29116162]
Average CV MAE: 10.694



--- Walk-forward retraining for year 2024 ---


TimeSeries CV MAE Scores: [22.93377554 11.36124033 15.76928509]
Average CV MAE: 16.688


In [6]:
print("Running Sensitivity Analysis (OAT)...")
class SensitivityAnalyzer:
    def __init__(self, model, df_engineered, feature_cols, dummy_cols):
        self.model = model
        self.df_engineered = df_engineered
        self.feature_cols = feature_cols
        self.dummy_cols = dummy_cols
        self.target_features = [
            "Births (thousands)",
            "Crude Birth Rate (births per 1,000 population)",
            "Infant Mortality Rate (infant deaths per 1,000 live births)",
            "Under-Five Mortality (deaths under age 5 per 1,000 live births)",
            "Net Number of Migrants (thousands)",
            "Net Migration Rate (per 1,000 population)",
            "Pop_Age_0(In Thousands)"
        ]

    def run_baseline(self, future_demo_df):
        return recursive_forecast(self.df_engineered, self.model, self.feature_cols, self.dummy_cols, 2025, TARGET, future_demo_df)

    def one_at_a_time(self, country, future_demo_df, perturbation_pct=5):
        baseline_pred = self.run_baseline(future_demo_df)
        country_baseline = baseline_pred[baseline_pred['Country'] == country].set_index('Year')['Predicted']
        results = []
        for feature in self.target_features:
            demo_high = future_demo_df.copy()
            country_mask = demo_high['Country'] == country
            if feature in demo_high.columns:
                demo_high.loc[country_mask, feature] *= (1 + perturbation_pct / 100.0)
                pred_high = recursive_forecast(self.df_engineered, self.model, self.feature_cols, self.dummy_cols, 2025, TARGET, demo_high)
                val_high = pred_high[pred_high['Country'] == country].set_index('Year')['Predicted']
                
                demo_low = future_demo_df.copy()
                demo_low.loc[country_mask, feature] *= (1 - perturbation_pct / 100.0)
                pred_low = recursive_forecast(self.df_engineered, self.model, self.feature_cols, self.dummy_cols, 2025, TARGET, demo_low)
                val_low = pred_low[pred_low['Country'] == country].set_index('Year')['Predicted']
                
                for year in country_baseline.index:
                    base = country_baseline[year]
                    high = val_high[year]
                    low = val_low[year]
                    impact_pct = ((high - base) / base) * 100 if base != 0 else 0
                    low_impact_pct = ((low - base) / base) * 100 if base != 0 else 0
                    results.append({
                        'Year': year,
                        'Feature': feature,
                        'High_Impact': impact_pct,
                        'Low_Impact': low_impact_pct,
                        'Abs_Impact': abs(impact_pct)
                    })
        return pd.DataFrame(results)

    def elasticity_curve(self, country, feature, future_demo_df, pct_range=(-20, 20), steps=21):
        baseline_pred = self.run_baseline(future_demo_df)
        country_baseline = baseline_pred[baseline_pred['Country'] == country].set_index('Year')['Predicted']
        pct_variations = np.linspace(pct_range[0], pct_range[1], steps)
        results = []
        for pct in pct_variations:
            demo_adj = future_demo_df.copy()
            country_mask = demo_adj['Country'] == country
            if feature in demo_adj.columns:
                demo_adj.loc[country_mask, feature] *= (1 + pct / 100.0)
                pred = recursive_forecast(self.df_engineered, self.model, self.feature_cols, self.dummy_cols, 2025, TARGET, demo_adj)
                val = pred[pred['Country'] == country].set_index('Year')['Predicted']
                for year in val.index:
                    results.append({
                        'Perturbation_Pct': pct,
                        'Year': year,
                        '% Change': ((val[year] - country_baseline[year]) / country_baseline[year]) * 100
                    })
        return pd.DataFrame(results)

analyzer = SensitivityAnalyzer(model, df_engineered, feature_cols, dummy_cols)

tornado_data = {}
feature_importance = {}

for c in COUNTRIES:
    oat = analyzer.one_at_a_time(c, future_demo_df, perturbation_pct=5)
    # Average across all forecast years
    oat_avg = oat.groupby('Feature').mean(numeric_only=True).reset_index()
    oat_avg = oat_avg.sort_values('Abs_Impact', ascending=False)
    
    tornado_data[c] = []
    feature_importance[c] = []
    
    for i, row in oat_avg.iterrows():
        feat = row['Feature']
        # Rename for cleaner UI
        name_map = {
            "Births (thousands)": "Births",
            "Crude Birth Rate (births per 1,000 population)": "Birth Rate",
            "Infant Mortality Rate (infant deaths per 1,000 live births)": "IMR",
            "Under-Five Mortality (deaths under age 5 per 1,000 live births)": "U5 Mortality",
            "Net Number of Migrants (thousands)": "Net Migration",
            "Net Migration Rate (per 1,000 population)": "Migration Rate",
            "Pop_Age_0(In Thousands)": "Pop Age 0"
        }
        short_name = name_map.get(feat, feat)
        
        tornado_data[c].append({
            "feature": short_name,
            "pos": round(row['High_Impact'], 2),
            "neg": round(row['Low_Impact'], 2)
        })
        
        # Feature importance
        score = row['Abs_Impact']
        impact_label = 'HIGH' if score > 0.5 else ('MEDIUM' if score > 0.2 else 'LOW')
        feature_importance[c].append({
            "name": short_name,
            "impact": impact_label,
            "score": round(score, 2)
        })

Running Sensitivity Analysis (OAT)...


In [7]:
print("Running Elasticity Analysis for top 3 features...")
elasticity_data = {}
for c in COUNTRIES:
    elasticity_data[c] = {}
    top_3_features = [f['name'] for f in feature_importance[c][:3]]
    
    # Reverse map short names to full column names
    reverse_map = {
        "Births": "Births (thousands)",
        "Birth Rate": "Crude Birth Rate (births per 1,000 population)",
        "IMR": "Infant Mortality Rate (infant deaths per 1,000 live births)",
        "U5 Mortality": "Under-Five Mortality (deaths under age 5 per 1,000 live births)",
        "Net Migration": "Net Number of Migrants (thousands)",
        "Migration Rate": "Net Migration Rate (per 1,000 population)",
        "Pop Age 0": "Pop_Age_0(In Thousands)"
    }
    
    for short_name in top_3_features:
        full_name = reverse_map.get(short_name, short_name)
        elast = analyzer.elasticity_curve(c, full_name, future_demo_df)
        
        # We'll plot elasticity for the final year (2030) as a representative curve
        elast_2030 = elast[elast['Year'] == 2030]
        
        pts = []
        for _, row in elast_2030.iterrows():
            pts.append({
                "x": round(row['Perturbation_Pct'], 1),
                "y": round(row['% Change'], 2)
            })
        elasticity_data[c][short_name] = pts

Running Elasticity Analysis for top 3 features...


In [8]:

from dataclasses import dataclass
from typing import Dict

@dataclass
class Scenario:
    name: str
    description: str
    adjustments: Dict[str, float]
    color: str

class ScenarioEngine:
    def __init__(self, model, df_engineered, feature_cols):
        self.model = model
        self.df_engineered = df_engineered
        self.feature_cols = feature_cols
        
        self.OPTIMISTIC = Scenario(
            name="optimistic",
            description="Strong health system, declining mortality, stable births",
            adjustments={
                "Infant Mortality Rate (infant deaths per 1,000 live births)": 0.85,
                "Under-Five Mortality (deaths under age 5 per 1,000 live births)": 0.85,
                "Births (thousands)": 1.02,
                "Pop_Age_0(In Thousands)": 1.02,
            },
            color="var(--green)"
        )
        
        self.PESSIMISTIC = Scenario(
            name="pessimistic", 
            description="Health system stress, rising mortality, declining coverage",
            adjustments={
                "Infant Mortality Rate (infant deaths per 1,000 live births)": 1.15,
                "Under-Five Mortality (deaths under age 5 per 1,000 live births)": 1.15,
                "Births (thousands)": 0.95,
                "Pop_Age_0(In Thousands)": 0.95,
            },
            color="var(--red)"
        )
        
        self.PANDEMIC_SHOCK = Scenario(
            name="pandemic",
            description="COVID-like disruption: coverage drops, migration halts",
            adjustments={
                "Net Number of Migrants (thousands)": 0.3,
                "Net Migration Rate (per 1,000 population)": 0.3,
                "Births (thousands)": 0.97,
                "Pop_Age_0(In Thousands)": 0.97,
            },
            color="var(--purple)"
        )
        
        self.BASELINE = Scenario(
            name="baseline",
            description="UN medium-variant projections (no adjustment)",
            adjustments={},
            color="var(--blue)"
        )
        
        self.default_scenarios = [self.BASELINE, self.OPTIMISTIC, self.PESSIMISTIC, self.PANDEMIC_SHOCK]
        
    def apply_scenario(self, demo_df, scenario):
        adjusted_df = demo_df.copy()
        for feature, multiplier in scenario.adjustments.items():
            if feature in adjusted_df.columns:
                adjusted_df[feature] *= multiplier
                if "Migration" not in feature and "Migrants" not in feature:
                    adjusted_df[feature] = adjusted_df[feature].clip(lower=0)
        return adjusted_df

    def run_all_scenarios(self, future_demo_df, scenarios=None):
        if scenarios is None:
            scenarios = self.default_scenarios
            
        results = {}
        for scenario in scenarios:
            adj_demo_df = self.apply_scenario(future_demo_df, scenario)
            pred = recursive_forecast(self.df_engineered, self.model, self.feature_cols, dummy_cols, 2025, TARGET, adj_demo_df)
            pred['Scenario'] = scenario.name
            results[scenario.name] = pred
            
        return results, scenarios

print("Running Scenario Engine...")
scen_engine = ScenarioEngine(model, df_engineered, feature_cols)
scenario_results, scenarios_list = scen_engine.run_all_scenarios(future_demo_df)

scenario_meta = {}
for s in scenarios_list:
    scenario_meta[s.name] = {
        "label": s.name.upper() if s.name != "pandemic" else "PANDEMIC SHOCK",
        "description": s.description,
        "color": s.color
    }

scenarios_data = {}
for c in COUNTRIES:
    scenarios_data[c] = {}
    for s in scenarios_list:
        s_df = scenario_results[s.name]
        c_pred = s_df[s_df['Country'] == c].sort_values('Year')['Predicted'].round(3).tolist()
        scenarios_data[c][s.name] = c_pred

print("Extracting Demographic Trends...")
demographic_trends = {}
for c in COUNTRIES:
    c_hist = df_raw[(df_raw['Country'] == c) & (df_raw['Year'] >= 2000) & (df_raw['Year'] <= 2024)].sort_values('Year')
    trends = []
    for _, row in c_hist.iterrows():
        trends.append({
            "year": int(row["Year"]),
            "pop": int(row.get("Total Population, as of 1 January (thousands)", 0) * 1000),
            "births": int(row.get("Births (thousands)", 0) * 1000),
            "br": round(row.get("Crude Birth Rate (births per 1,000 population)", 0), 2),
            "imr": round(row.get("Infant Mortality Rate (infant deaths per 1,000 live births)", 0), 2),
            "u5": round(row.get("Under-Five Mortality (deaths under age 5 per 1,000 live births)", 0), 2),
            "mig": int(row.get("Net Number of Migrants (thousands)", 0) * 1000)
        })
    demographic_trends[c] = trends


print("Assembling final JSON and exporting...")
project_dir = os.path.dirname(backend_dir)
frontend_public_dir = os.path.join(project_dir, 'frontend', 'public')
os.makedirs(frontend_public_dir, exist_ok=True)

data = {
    "countries": COUNTRIES,
    "historical": {},
    "forecastBaseline": forecast_baseline,
    "demographics": {},
    "mcData": mc_data,
    "tornadoData": tornado_data,
    "featureImportance": feature_importance,
    "elasticityData": elasticity_data,
    "backtest": backtest_data,
    "scenarios": scenarios_data,
    "scenarioMeta": scenario_meta,
    "demographicTrends": demographic_trends
}

# Add Historical Data
for c in COUNTRIES:
    hist = df_raw[(df_raw['Country'] == c) & (df_raw['Year'] >= 2000) & (df_raw['Year'] <= 2024)].sort_values('Year')
    data["historical"][c] = hist[['Year', TARGET]].rename(columns={TARGET: 'value', 'Year': 'year'}).to_dict('records')

# Add Demographics
for c in COUNTRIES:
    demo_row = future_demo_df[(future_demo_df['Country'] == c) & (future_demo_df['Year'] == 2025)].iloc[0]
    pop_0 = demo_row['Pop_Age_0(In Thousands)']
    age_dist = [
        {'age': 'Age 0', 'value': round(pop_0, 1)},
        {'age': 'Age 1', 'value': round(pop_0 * 0.99, 1)},
        {'age': 'Age 2', 'value': round(pop_0 * 0.98, 1)},
        {'age': 'Age 3', 'value': round(pop_0 * 0.97, 1)},
        {'age': 'Age 4', 'value': round(pop_0 * 0.96, 1)},
        {'age': 'Age 5', 'value': round(pop_0 * 0.95, 1)},
    ]
    data["demographics"][c] = {
        "totalPopulation": int(demo_row['Total Population, as of 1 January (thousands)'] * 1000),
        "births2025": int(demo_row['Births (thousands)'] * 1000),
        "crudeBirthRate": round(demo_row['Crude Birth Rate (births per 1,000 population)'], 2),
        "infantMortalityRate": round(demo_row['Infant Mortality Rate (infant deaths per 1,000 live births)'], 2),
        "under5Mortality": round(demo_row['Under-Five Mortality (deaths under age 5 per 1,000 live births)'], 2),
        "netMigration": int(demo_row['Net Number of Migrants (thousands)'] * 1000),
        "ageDistribution2025": age_dist
    }

out_path = os.path.join(frontend_public_dir, 'data.json')
with open(out_path, 'w') as f:
    json.dump(data, f, indent=2)

print(f"Data successfully generated at {out_path}")

Running Scenario Engine...


Extracting Demographic Trends...
Assembling final JSON and exporting...
Data successfully generated at D:\Home\Desktop\mcv1_final - Copy\frontend\public\data.json
